In [ ]:
import gc, json, sys, time
from pathlib import Path
import torch

root=Path('/kaggle/input')
manifests=list(root.rglob('phase6_data_manifest.json'))
if len(manifests)!=1:
    raise RuntimeError(f'Expected one official Phase 6 manifest; found {len(manifests)}')
data_root=manifests[0].parent
parents=list(root.rglob('a0_phase1_model.pt'))
if len(parents)!=1:
    raise RuntimeError(f'Expected one own Phase 1 A0 checkpoint; found {len(parents)}')
models_root=parents[0].parent
teacher_adapters=list(root.rglob('best_adapter.pt'))
if len(teacher_adapters)!=1:
    raise RuntimeError(f'Expected one completed own B1 teacher adapter; found {len(teacher_adapters)}')
if not torch.cuda.is_available():
    raise RuntimeError('A0 KD requires the requested GPU runtime')
sys.path.insert(0,str(data_root))
print(json.dumps({'event':'inputs_ready','gpu':torch.cuda.get_device_name(0),
                  'data_root':str(data_root),'models_root':str(models_root),
                  'teacher_adapter':str(teacher_adapters[0])}),flush=True)

# Teacher outputs are generated only for the deterministic fitting split.
from prepare_teacher_summaries import main as prepare_teacher
teacher_path=Path('/kaggle/working/teacher_predictions.json')
sys.argv=['prepare_teacher_summaries.py','--models-input',str(models_root),
          '--data-input',str(data_root),'--teacher-adapter',str(teacher_adapters[0]),
          '--output',str(teacher_path),'--count','256','--max-new-tokens','512']
prepare_teacher()
teacher_map=json.loads(teacher_path.read_text(encoding='utf-8'))
manifest=json.loads(teacher_path.with_suffix('.manifest.json').read_text(encoding='utf-8'))
if len(teacher_map)!=256 or manifest.get('fit_only') is not True:
    raise RuntimeError('Teacher generation count/split provenance check failed')
print(json.dumps({'event':'teacher_targets_verified','count':len(teacher_map),
                  'fit_only':manifest['fit_only'],
                  'teacher_parameters':manifest['teacher_model_parameters'],
                  'student_parameters':manifest['student_phase1_parameters']}),flush=True)

gc.collect(); torch.cuda.empty_cache()
# Student uses its own Phase 1 checkpoint/tokenizer. Text-level targets bridge the
# B1/A0 tokenizer difference; no incompatible vocabulary KL is used.
from train_phase6 import main as train_student
sys.argv=['train_phase6.py','--input',str(data_root),'--models-input',str(models_root),
          '--output','/kaggle/working/a0_kd','--method','a0_kd',
          '--epochs','4','--accum','8','--lr','0.00015',
          '--passages','2','--passage-tokens','384','--target-tokens','1024',
          '--validation-limit','16','--confirmation-limit','32',
          '--max-train-hours','3','--skip-bertscore',
          '--teacher-predictions',str(teacher_path)]
train_student()
